In [3]:
from diagrams import Diagram, Cluster
from diagrams.custom import Custom
from diagrams.programming.language import Python
#from diagrams.onprem.ml import Tensorflow
from diagrams.onprem.database import PostgreSQL

In [8]:
with Diagram("Search and Ranking Pipeline", direction="TB"):

    # Input vectors
    user_query = Custom("User Query\n($\\mathbf{q}$)", "./icons/vector.png")
    hard_keys = Custom("Hard Keys\n($\\mathbf{k}$)", "./icons/filter.png")
    user_details = Custom("User Details\n($\\mathbf{u}$)", "./icons/user.png")
    liked_products = Custom("Liked Products\n($\\mathbf{p}$)", "./icons/like.png")
    aesthetics = Custom("Aesthetic Matrix\n($\\mathbf{A}$)", "./icons/aesthetic.png")

    # Stage 1: Filtering
    with Cluster("1. Filtering Stage"):
        filtering = Python("Filter(V_DB, k)")

    # Stage 2: KNN Similarity Search
    with Cluster("2. KNN Similarity Search"):
        embedding_calc = Python("v_emb = αq + βu + γp + Σδ_iA_i")
        knn = Python("KNN Search (v_emb)")

    # Stage 3: Ranking Model
    with Cluster("3. Ranking Model (CNN)"):
        sim_matrix = Python("S = similarity matrix")
        cnn_rank = Python("CNN Relevance Scorer")

    # Stage 4: Ranking
    with Cluster("4. Product Ranking"):
        sorting = Python("Sort by Relevance")
        top_r = Python("Select Top r")

    # Stage 5: Retrieval
    retrieve = PostgreSQL("Relational DB\nProduct Info")
    output = Custom("Ranked Product List", "./icons/output.png")

    # Connections
    [user_query, hard_keys, user_details, liked_products, aesthetics] >> filtering
    [user_query, user_details, liked_products, aesthetics] >> embedding_calc
    filtering >> knn
    embedding_calc >> knn >> sim_matrix >> cnn_rank >> sorting >> top_r >> retrieve >> output

In [ ]:
# Create a cleaner, more detailed pipeline diagram using Graphviz
import graphviz

# Create a new digraph
g = graphviz.Digraph('search_ranking_pipeline', 
                     format='png',
                     engine='dot',
                     strict=False)

# Set graph attributes for overall layout and style
g.attr(rankdir='TB',  # Top to Bottom layout
       nodesep='0.5',
       ranksep='0.8',
       fontname='Arial', 
       fontsize='16',
       bgcolor='white',
       label='Search and Ranking Pipeline',
       labelloc='t',
       fontcolor='#333333',
       size='11,14',  # Width, height in inches
       dpi='300',
       compound='true')

# Node styles
g.attr('node', shape='box', style='rounded,filled', fontname='Arial', fontsize='12', 
       fillcolor='#F5F5F5', color='#333333', margin='0.2,0.1', height='0.6')

# Edge styles
g.attr('edge', fontname='Arial', fontsize='10', color='#666666', arrowsize='0.7')

# User & Context Inputs cluster
with g.subgraph(name='cluster_inputs') as c:
    c.attr(label='User & Context Inputs', style='rounded,filled', fillcolor='#E1F5FE', color='#01579B', fontcolor='#01579B')
    c.node('user_query', 'User Query\n(q)', shape='ellipse', fillcolor='#B3E5FC')
    c.node('hard_keys', 'Hard Keys\n(k)', shape='ellipse', fillcolor='#B3E5FC')
    c.node('user_details', 'User Details\n(u)', shape='ellipse', fillcolor='#B3E5FC')
    c.node('liked_products', 'Liked Products\n(p)', shape='ellipse', fillcolor='#B3E5FC')
    c.node('aesthetics', 'Aesthetic Matrix\n(A)', shape='ellipse', fillcolor='#B3E5FC')
    c.attr(rank='same')

# Vector Database cluster
with g.subgraph(name='cluster_vector_db') as c:
    c.attr(label='Vector Database', style='rounded,filled', fillcolor='#E8F5E9', color='#2E7D32', fontcolor='#2E7D32')
    c.node('vector_db', 'Product Vector DB', shape='cylinder', fillcolor='#C8E6C9')

# Stage 1: Filtering cluster
with g.subgraph(name='cluster_filtering') as c:
    c.attr(label='1. Filtering Stage', style='rounded,filled', fillcolor='#FFF3E0', color='#E65100', fontcolor='#E65100')
    c.node('filtering', 'Filter(V_DB, k)', fillcolor='#FFE0B2')

# Stage 2: Vector Embedding & KNN Search cluster
with g.subgraph(name='cluster_embedding_knn') as c:
    c.attr(label='2. Vector Embedding & KNN Search', style='rounded,filled', fillcolor='#F3E5F5', color='#6A1B9A', fontcolor='#6A1B9A')
    c.node('embedding_calc', 'v_emb = αq + βu + γp + Σδ_iA_i', fillcolor='#E1BEE7')
    c.node('knn', 'KNN Search\n(Top N similar products)', fillcolor='#E1BEE7')
    g.edge('embedding_calc', 'knn')

# Stage 3: Relevance Scoring cluster
with g.subgraph(name='cluster_relevance') as c:
    c.attr(label='3. Relevance Scoring', style='rounded,filled', fillcolor='#E8EAF6', color='#1A237E', fontcolor='#1A237E')
    c.node('sim_matrix', 'Similarity Matrix (S)\nProduct vs. all input vectors', fillcolor='#C5CAE9')
    c.node('cnn_rank', 'CNN Relevance Scorer\nOutputs r ∈ ℝᴺ', fillcolor='#C5CAE9')
    g.edge('sim_matrix', 'cnn_rank')

# Stage 4: Final Processing cluster
with g.subgraph(name='cluster_final_processing') as c:
    c.attr(label='4. Final Processing', style='rounded,filled', fillcolor='#E0F2F1', color='#00695C', fontcolor='#00695C')
    c.node('sorting', 'Sort by Relevance Score', fillcolor='#B2DFDB')
    c.node('top_r', 'Select Top r Results', fillcolor='#B2DFDB')
    g.edge('sorting', 'top_r')

# Stage 5: Product Retrieval cluster
with g.subgraph(name='cluster_retrieval') as c:
    c.attr(label='5. Product Retrieval', style='rounded,filled', fillcolor='#FFFDE7', color='#F9A825', fontcolor='#F9A825')
    c.node('retrieve', 'Relational DB\nComplete Product Info', shape='cylinder', fillcolor='#FFF9C4')
    c.node('output', 'Ranked Product List', shape='note', fillcolor='#FFECB3')
    g.edge('retrieve', 'output')

# Add a note
g.node('note', 'Note: This diagram represents the multi-stage search and\nranking pipeline that combines vector similarity search\nwith neural network-based relevance scoring.',
       shape='note', fillcolor='#FFF8E1', style='filled', fontcolor='#5D4037')

# Main connections - Inputs to processing stages
g.edge('user_query', 'embedding_calc')
g.edge('user_details', 'embedding_calc')
g.edge('liked_products', 'embedding_calc')
g.edge('aesthetics', 'embedding_calc')

g.edge('vector_db', 'filtering')
g.edge('hard_keys', 'filtering')

# Main pipeline flow connections
g.edge('filtering', 'knn')
g.edge('knn', 'sim_matrix')
g.edge('cnn_rank', 'sorting')
g.edge('top_r', 'retrieve')

# Render and display the diagram
g.render('search_ranking_pipeline_graphviz', format='png', cleanup=True, view=True)
display(g)